# UbuntuCare Online Smoke Test

Run all cells in order. **Managed Colab runtimes with read-only cgroups automatically use remote mode**: they test an existing UbuntuCare HTTPS deployment and do not attempt Docker installation. This notebook cannot provision a VM or make Docker work inside a restricted Colab runtime.

For a temporary local demo, use a Docker-enabled Linux VM with writable cgroups. The notebook clones `rager1904/beverly-eye` (`main`) and launches frontend, gateway, EMR, PostgreSQL, Redis, Keycloak, and migrations. AI services are omitted. To test local changes, supply an existing `REPO_DIR` Path before the setup cell instead of cloning.

**Synthetic data only.** Local mode exposes the login page via a temporary public Cloudflare tunnel. Keep credentials private. Write tests are disabled; cleanup is opt-in so Run all does not immediately shut down the demo. This is a smoke test, not a production deployment or a full AI evaluation.

Local demo logins use the shared password `UbuntuCare2024!`: admin `admin@ubuntucare.gov.zm` and doctor `john.mwamba@uh.gov.zm`. Use a fresh runtime to apply this to an already-created demo. Remote-mode credentials are managed by that deployment. Never use this shared password with real patient data.


In [ ]:
%pip -q install requests
import requests

## 0. Select runtime and connect

Leave `LAUNCH_MODE = 'auto'` for Colab. On restricted runtimes the launch cells print a skip message and proceed to the login cell, which needs an existing public HTTPS origin, hospital UUID, and login credentials. Remote mode does not create a deployment. Use `remote` explicitly to skip all installation. Use `local` only on a Docker-enabled Linux VM; package installation requires root and host ports 3000/5432/6379/8000/8001/8080 must be free.

In local mode use the email/password login form; SSO is not configured for the single frontend tunnel. A failed real dependency check stops execution instead of reporting a false pass.

In [ ]:
from pathlib import Path
import os, shutil, subprocess, tempfile, time

# auto: use Docker only where cgroups are writable; otherwise test a remote deployment.
# remote: never clone/install/start containers. local: require a Docker-enabled Linux VM.
LAUNCH_MODE = 'auto'  # auto | remote | local
REMOTE_FRONTEND_URL = ''  # Existing HTTPS deployment URL; prompted below when empty.
RUN_WRITE_TESTS = False
STOP_DEMO = False  # Set True and rerun the final cell to close the temporary demo.

def cgroup_mount_is_read_only(mountinfo=None):
    if mountinfo is None:
        try:
            mountinfo = Path('/proc/self/mountinfo').read_text()
        except OSError:
            return False
    for line in mountinfo.splitlines():
        before, separator, after = line.partition(' - ')
        fields, filesystem = before.split(), after.split()
        if separator and len(fields) > 5 and filesystem and fields[4].startswith('/sys/fs/cgroup') and filesystem[0] in {'cgroup', 'cgroup2'}:
            if 'ro' in fields[5].split(','):
                return True
    return False

if LAUNCH_MODE not in {'auto', 'remote', 'local'}:
    raise ValueError('LAUNCH_MODE must be auto, remote, or local')
LOCAL_LAUNCH = LAUNCH_MODE != 'remote' and os.name == 'posix' and not cgroup_mount_is_read_only()
if LAUNCH_MODE == 'local' and not LOCAL_LAUNCH:
    raise RuntimeError('Local Docker needs a Linux VM with writable cgroups. Select auto or remote and supply an existing deployment URL.')
if not LOCAL_LAUNCH:
    print('Remote mode: skipping Docker installation, builds, and tunnel. Supply an existing UbuntuCare HTTPS URL at the login cell.')

if LOCAL_LAUNCH:
    if shutil.which('git') is None:
        subprocess.run(['apt-get', 'update', '-qq'], check=True)
        subprocess.run(['apt-get', 'install', '-y', '-qq', 'git'], check=True)

    REPO_URL = 'https://github.com/rager1904/beverly-eye.git'
    BRANCH = 'main'
    if 'REPO_DIR' not in globals():
        REPO_DIR = Path(tempfile.mkdtemp(prefix='beverly-eye-colab-')) / 'repo'
    if not (REPO_DIR / '.git').exists():
        clone = subprocess.run(
            ['git', 'clone', '--depth', '1', '--branch', BRANCH, REPO_URL, str(REPO_DIR)],
            text=True, capture_output=True,
        )
        if clone.returncode:
            print(clone.stderr[-4000:])
            raise RuntimeError('Could not clone Beverly Eye from GitHub main; see git error above.')
    if not (REPO_DIR / 'src/frontend/web/package.json').exists():
        raise FileNotFoundError(f'Unexpected repository contents at {REPO_DIR}')
    print('Cloned branch main to:', REPO_DIR)

In [ ]:
if LOCAL_LAUNCH:
    WORK_DIR = Path(tempfile.gettempdir()) / 'ubuntucare-notebook'
    WORK_DIR.mkdir(mode=0o700, parents=True, exist_ok=True)
    import os, shutil, time

    def run_cmd(args, **kwargs):
        return subprocess.run(args, check=True, text=True, **kwargs)

    subprocess.run(['apt-get', 'update', '-qq'], check=True)
    if shutil.which('docker') is None or shutil.which('dockerd') is None:
        run_cmd(['apt-get', 'install', '-y', '-qq', 'docker.io', 'curl', 'git'])
    else:
        run_cmd(['apt-get', 'install', '-y', '-qq', 'curl', 'git'])

    COMPOSE_CMD = ['docker', 'compose']
    plugin_dir = Path('/usr/local/lib/docker/cli-plugins')
    plugin_dir.mkdir(parents=True, exist_ok=True)
    plugin = plugin_dir / 'docker-compose'
    run_cmd(['curl', '-fsSL', 'https://github.com/docker/compose/releases/download/v2.32.4/docker-compose-linux-x86_64', '-o', str(plugin)])
    plugin.chmod(0o755)

    if shutil.which('cloudflared') is None:
        run_cmd(['curl', '-fsSL', 'https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64', '-o', '/usr/local/bin/cloudflared'])
        Path('/usr/local/bin/cloudflared').chmod(0o755)

    docker_check = subprocess.run(['docker', 'info'], capture_output=True, text=True)
    if docker_check.returncode != 0:
        DOCKER_LOG_PATH = WORK_DIR / 'dockerd.log'
        DOCKER_LOG = DOCKER_LOG_PATH.open('w')
        # Colab commonly denies bridge/iptables operations. Host mode in the
        # Colab Compose override avoids creating Docker bridge networks.
        DOCKER_DAEMON = subprocess.Popen([
            'dockerd', '--host=unix:///var/run/docker.sock', '--iptables=false',
            '--bridge=none', '--ip-forward=false', '--ip-masq=false',
            '--storage-driver=vfs',
        ], stdout=DOCKER_LOG, stderr=subprocess.STDOUT, start_new_session=True)
        for _ in range(90):
            if subprocess.run(['docker', 'info'], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL).returncode == 0:
                break
            if DOCKER_DAEMON.poll() is not None:
                tail = '\n'.join(DOCKER_LOG_PATH.read_text(errors='ignore').splitlines()[-40:])
                raise RuntimeError('Docker daemon stopped; this Colab runtime may disallow nested Docker. Recent dockerd log:\n' + tail)
            time.sleep(2)
        else:
            tail = '\n'.join(DOCKER_LOG_PATH.read_text(errors='ignore').splitlines()[-40:])
            raise RuntimeError('Docker daemon did not become ready. Recent dockerd log:\n' + tail)

    print(subprocess.run(COMPOSE_CMD + ['version'], capture_output=True, text=True).stdout.strip())
    print('Docker engine is ready.')
else:
    print('Skipped Docker setup (remote mode).')

In [ ]:
if LOCAL_LAUNCH:
    import secrets
    import string
    from uuid import uuid4

    def make_demo_password():
        # Meet the Keycloak demo realm's minimum length/complexity policy.
        return 'Colab-7' + secrets.token_hex(12) + '!'

    if 'COLAB_PROJECT' not in globals():
        DEMO_LOGIN_EMAIL = 'admin@ubuntucare.gov.zm'
        DEMO_LOGIN_PASSWORD = 'UbuntuCare2024!'  # Shared synthetic-demo login only.
        env = {
            'POSTGRES_USER': 'ubuntucare',
            'POSTGRES_DB': 'ubuntucare',
            'POSTGRES_PASSWORD': secrets.token_urlsafe(32),
            'SECRET_KEY': secrets.token_urlsafe(48),
            'KEYCLOAK_ADMIN_USER': 'admin',
            'KEYCLOAK_ADMIN_PASSWORD': make_demo_password(),
            'KEYCLOAK_DEMO_USER_PASSWORD': DEMO_LOGIN_PASSWORD,
            'KEYCLOAK_REALM': 'ubuntucare',
            'KEYCLOAK_CLIENT_ID': 'api-gateway',
            'KEYCLOAK_CLIENT_SECRET': secrets.token_urlsafe(36),
            'KEYCLOAK_FRONTEND_CLIENT_ID': 'frontend',
            'KEYCLOAK_HOSTNAME': 'keycloak',
            'MINIO_ROOT_USER': 'ubuntucare',
            'MINIO_ROOT_PASSWORD': secrets.token_urlsafe(32),
            'FLOWER_PASSWORD': secrets.token_urlsafe(24),
            'ELASTIC_PASSWORD': secrets.token_urlsafe(24),
            'GRAFANA_ADMIN_PASSWORD': secrets.token_urlsafe(24),
            'FRONTEND_PORT': '3000',
            'API_GATEWAY_PORT': '8000',
            'EMR_PORT': '8001',
        }
        COLAB_ENV_FILE = WORK_DIR / 'ubuntucare-colab.env'
        COLAB_ENV_FILE.write_text('\n'.join(f'{key}={value}' for key, value in env.items()) + '\n', encoding='utf-8')
        COLAB_ENV_FILE.chmod(0o600)
        COLAB_PROJECT = f'ubuntucare-colab-{uuid4().hex[:8]}'
    COMPOSE_FILES = ['-f', 'docker-compose.yml', '-f', 'docs/colab/docker-compose.colab.yml']
    COMPOSE = COMPOSE_CMD + ['-p', COLAB_PROJECT, '--env-file', str(COLAB_ENV_FILE)] + COMPOSE_FILES

    # Starting only these targets starts their core dependencies; Compose AI profiles remain off.
    print('Building and starting frontend + API gateway + EMR and required core dependencies...')
    COMPOSE_LOG_PATH = WORK_DIR / 'ubuntucare-compose-up.log'
    compose_result = subprocess.run(
        COMPOSE + ['--progress', 'plain', 'up', '-d', '--build', 'frontend', 'api-gateway', 'emr'],
        cwd=REPO_DIR, capture_output=True, text=True, timeout=1800,
    )
    compose_output = (compose_result.stdout or '') + '\n' + (compose_result.stderr or '')
    COMPOSE_LOG_PATH.write_text(compose_output, encoding='utf-8')
    if compose_result.returncode:
        print('Compose build/start failed; final output follows:\n')
        print('\n'.join(compose_output.splitlines()[-100:]))
        ps_result = subprocess.run(COMPOSE + ['ps', '-a'], cwd=REPO_DIR, capture_output=True, text=True)
        if ps_result.stdout:
            print('\nContainer status:\n' + ps_result.stdout[-8000:])
        logs_result = subprocess.run(COMPOSE + ['logs', '--no-color', '--tail=100'], cwd=REPO_DIR, capture_output=True, text=True)
        if logs_result.stdout:
            print('\nService logs:\n' + logs_result.stdout[-12000:])
        raise RuntimeError(f'docker compose up failed with exit {compose_result.returncode}. Full output: {COMPOSE_LOG_PATH}')
    print('\n'.join(compose_output.splitlines()[-30:]))
    print('Core stack is up. No ASR, TTS, RAG, or agent-worker services were started.')
else:
    print('Skipped stack startup (remote mode).')

In [ ]:
if LOCAL_LAUNCH:
    import re
    from IPython.display import HTML, display

    # Wait for the login page before opening a temporary HTTPS tunnel.
    for _ in range(90):
        try:
            local_page = requests.get('http://127.0.0.1:3000/login', timeout=5)
            if local_page.status_code == 200:
                break
        except requests.RequestException:
            pass
        time.sleep(2)
    else:
        raise RuntimeError('Frontend did not become ready. Check: docker compose logs frontend emr api-gateway')

    TUNNEL_LOG_PATH = WORK_DIR / 'cloudflared.log'
    TUNNEL_LOG = TUNNEL_LOG_PATH.open('w')
    if 'CLOUDFLARED' not in globals() or CLOUDFLARED.poll() is not None:
        CLOUDFLARED = subprocess.Popen(['cloudflared', 'tunnel', '--no-autoupdate', '--url', 'http://127.0.0.1:3000'], stdout=TUNNEL_LOG, stderr=subprocess.STDOUT, start_new_session=True)
    FRONTEND_URL = None
    for _ in range(90):
        log_text = TUNNEL_LOG_PATH.read_text(errors='ignore')
        match = re.search(r'https://[a-z0-9-]+\.trycloudflare\.com', log_text)
        if match:
            FRONTEND_URL = match.group(0)
            break
        if CLOUDFLARED.poll() is not None:
            raise RuntimeError('Cloudflare tunnel stopped. Check /content/cloudflared.log.')
        time.sleep(2)
    if not FRONTEND_URL:
        raise RuntimeError('Could not obtain a Cloudflare URL. Check /content/cloudflared.log.')
    BASE_URL = FRONTEND_URL
    print('Frontend login:', FRONTEND_URL + '/login')
    print('Demo login email:', DEMO_LOGIN_EMAIL)
    print('Demo password is stored in DEMO_LOGIN_PASSWORD; inspect it privately only when needed.')
    print('Use the email/password form. Do not click SSO in this single-tunnel demo.')
    display(HTML(f'<p><a href="{FRONTEND_URL}/login" target="_blank"><strong>Open UbuntuCare login</strong></a></p>'))
    print('Do not save/share notebook outputs: the temporary URL and demo password are visible here.')
else:
    print('Skipped public tunnel (remote mode).')

In [ ]:
from getpass import getpass
from datetime import datetime, timezone
from uuid import uuid4
from urllib.parse import urlparse
import requests

# Never reuse a stale local URL or local credentials for a remote deployment.
BASE_URL = (FRONTEND_URL if LOCAL_LAUNCH else (REMOTE_FRONTEND_URL or input('Public UbuntuCare HTTPS URL: '))).strip().rstrip('/')
parsed = urlparse(BASE_URL)
if parsed.scheme != 'https' or not parsed.hostname or parsed.username or parsed.password or parsed.query or parsed.fragment or parsed.path not in {'', '/'}:
    raise ValueError('Use an HTTPS origin such as https://demo.example.com, without credentials, path, query, or fragment.')
if parsed.hostname in {'localhost', '127.0.0.1', '::1'}:
    raise ValueError('Use a public deployment URL reachable from Colab.')
HOSPITAL_ID = str(__import__('uuid').UUID(input('Hospital UUID [00000000-0000-4000-8000-000000000001]: ').strip() or '00000000-0000-4000-8000-000000000001'))

session = requests.Session()
session.headers.update({'Accept': 'application/json'})
API = f'{BASE_URL}/api/v1'

if LOCAL_LAUNCH:
    LOGIN_EMAIL, LOGIN_PASSWORD = DEMO_LOGIN_EMAIL, DEMO_LOGIN_PASSWORD
else:
    LOGIN_EMAIL = input('UbuntuCare login email: ').strip()
    LOGIN_PASSWORD = getpass('UbuntuCare password (hidden): ')
login_response = session.post(f'{API}/auth/login', json={'email': LOGIN_EMAIL, 'password': LOGIN_PASSWORD}, timeout=30, allow_redirects=False)
print('POST /auth/login:', login_response.status_code)
login_response.raise_for_status()
if login_response.status_code != 200:
    raise RuntimeError('Login must return HTTP 200 without redirects; verify the deployment URL.')
ACCESS_TOKEN = login_response.json().get('access_token')
if not isinstance(ACCESS_TOKEN, str) or not ACCESS_TOKEN:
    raise RuntimeError('Login returned no access token.')
del LOGIN_PASSWORD
login_response = None
session.headers['Authorization'] = f'Bearer {ACCESS_TOKEN}'
session.headers['X-Hospital-ID'] = HOSPITAL_ID
print('Authenticated as:', LOGIN_EMAIL)
print('Configured deployment:', BASE_URL)
print('Write tests enabled:', RUN_WRITE_TESTS)

## 1. Gateway health (read-only)

This request does not need a token. A `healthy` result means the gateway can reach its database and Redis checks; other dependencies may still need separate attention.

In [ ]:
health_url = 'http://127.0.0.1:8000/health' if LOCAL_LAUNCH else f'{BASE_URL}/health'
health_response = requests.get(health_url, timeout=30, allow_redirects=False)
print('GET /health:', health_response.status_code)
if health_response.status_code == 404 and not LOCAL_LAUNCH:
    print('Gateway health is not proxied by this frontend URL; continuing with authenticated API checks.')
else:
    if health_response.status_code != 200:
        raise RuntimeError(f'Gateway health failed: HTTP {health_response.status_code}')
    health_response.raise_for_status()
    health = health_response.json()
    print('Gateway status:', health.get('status'))
    print('Database:', health.get('checks', {}).get('database', {}).get('status'))
    print('Redis:', health.get('checks', {}).get('redis', {}).get('status'))
    if health.get('status') != 'healthy':
        raise RuntimeError('Gateway reports unhealthy dependencies.')

## 2. Authenticated read-only checks

The notebook logs in using the credentials above. The temporary stack uses a generated demo admin account. The notebook reports status/counts only; it avoids printing patient records.

In [ ]:
def get_json(path, params=None):
    response = session.get(f'{API}{path}', params=params, timeout=30, allow_redirects=False)
    print(f'GET {path}:', response.status_code)
    if response.status_code != 200:
        raise RuntimeError(f'GET {path} failed: HTTP {response.status_code}. Check URL, tenant, role, and token expiry.')
    return response.json()

departments_data = get_json('/hospital/departments')
departments = departments_data if isinstance(departments_data, list) else departments_data.get('items', [])
active_department_names = {d['name'] for d in departments if d.get('is_active', True) and isinstance(d.get('name'), str)}
print('Active departments:', sorted(active_department_names))
expected_departments = {'General', 'Eye', 'Dental'}
assert expected_departments.issubset(active_department_names), f'Missing required departments; expected {sorted(expected_departments)}, got {sorted(active_department_names)}'

staff_data = get_json('/hospital/staff', params={'per_page': 100})
staff_items = staff_data.get('items', []) if isinstance(staff_data, dict) else staff_data
print('Directory staff records available:', len(staff_items))

destinations = get_json('/patient-flow/destinations')
service_points = {item.get('code') for item in destinations.get('service_points', [])}
print('Core service destinations present:', sorted({'general', 'eye', 'dental'} & service_points))
assert {'general', 'eye', 'dental'}.issubset(service_points), 'General/Eye/Dental patient-flow destinations are missing'

# A unique synthetic query checks patient search without listing or displaying real patients.
search_marker = f'COLAB-NONEXISTENT-{uuid4().hex[:10]}'
patient_search = get_json('/emr/patients', params={'query': search_marker, 'per_page': 5})
print('Synthetic patient search completed; matches:', patient_search.get('total', 0))

notes_data = get_json('/emr/notes', params={'patient_id': str(uuid4()), 'page': 1, 'per_page': 1})
print('Clinical-note API accessible; total notes:', notes_data.get('total', 'not reported'))
print('Read-only checks passed.')

## 3. Optional write test — disabled by default

Only continue in the temporary Colab database or a **test/staging environment**. Set `RUN_WRITE_TESTS = True` in the configuration cell, then run this cell. It creates one clearly labelled synthetic patient, an Eye visit with NHIMA/review/paid test values, and a draft eye SOAP note. These records remain in the database; the notebook does not delete records. Do not use real names, NRCs, phone numbers, or clinical information.

In [ ]:
if not RUN_WRITE_TESTS:
    print('Skipped: set RUN_WRITE_TESTS = True only after confirming this is a test/staging deployment.')
else:
    marker = uuid4().hex[:8].upper()
    eye_department = next((d for d in departments if d.get('name') == 'Eye' and d.get('is_active', True)), None)
    if eye_department is None:
        raise RuntimeError('Run the read-only checks first: active Eye department is required before writing data.')
    patient_payload = {
        'first_name': 'ColabTest',
        'last_name': f'Synthetic{marker}',
        'date_of_birth': '1990-01-01',
        'gender': 'other',
        'phone': '+260000000000',
        'notes': 'SYNTHETIC COLAB SMOKE TEST DATA — not a real patient.',
    }
    patient_response = session.post(f'{API}/emr/patients', json=patient_payload, timeout=30, allow_redirects=False)
    print('POST /emr/patients:', patient_response.status_code)
    if patient_response.status_code != 201:
        raise RuntimeError('Creating patient failed: HTTP ' + str(patient_response.status_code) + '. Earlier synthetic records may remain; do not rerun blindly.')
    patient_id = patient_response.json()['id']

    eye_department = next((d for d in departments if d.get('name') == 'Eye' and d.get('is_active', True)), None)
    journey_payload = {
        'patient_id': patient_id,
        'priority': 'routine',
        'payment_scheme': 'nhima',
        'visit_type': 'review',
        'payment_status': 'paid',
        'scheme_number': f'COLAB-{marker}',
        'chief_complaint': 'SYNTHETIC TEST DATA ONLY — routine eye review',
        'step_type': 'consultation',
        'service_point': 'eye',
        'department_id': eye_department.get('id') if eye_department else None,
        'department_name': 'Eye',
    }
    journey_response = session.post(f'{API}/patient-flow/journeys', json=journey_payload, timeout=30, allow_redirects=False)
    print('POST /patient-flow/journeys:', journey_response.status_code)
    if journey_response.status_code != 201:
        raise RuntimeError('Creating journey failed: HTTP ' + str(journey_response.status_code) + '. Earlier synthetic records may remain; do not rerun blindly.')
    journey = journey_response.json()
    print('Created synthetic journey:', journey.get('journey_number'), '|', journey.get('visit_type'), '|', journey.get('payment_status'))

    note_payload = {
        'patient_id': patient_id,
        'note_type': 'SOAP',
        'title': f'Colab Smoke Test - Eye Note {marker}',
        'subjective': 'SYNTHETIC TEST DATA ONLY — no real patient symptoms.',
        'objective': 'SYNTHETIC TEST DATA ONLY — no examination performed.',
        'assessment': 'Test record only; not a clinical diagnosis.',
        'plan': 'Test record only; no treatment.',
        'authored_by': 'Colab Smoke Test',
        'authored_datetime': datetime.now(timezone.utc).isoformat(),
        'status': 'draft',
    }
    note_response = session.post(f'{API}/emr/notes', json=note_payload, timeout=30, allow_redirects=False)
    print('POST /emr/notes:', note_response.status_code)
    if note_response.status_code != 201:
        raise RuntimeError('Creating note failed: HTTP ' + str(note_response.status_code) + '. Earlier synthetic records may remain; do not rerun blindly.')
    print('Created synthetic draft note:', note_response.json().get('id'))
    print('Write smoke test passed. Synthetic records were left in the test database.')

## 4. Stop the temporary demo (opt-in)

Set `STOP_DEMO = True` in the final cell, then run it when finished. It closes the tunnel and removes the notebook-managed containers, retaining database volumes. Remote deployments are not stopped. Run all leaves the demo available for manual browser checks.

In [ ]:
if not STOP_DEMO:
    print('Cleanup skipped. Set STOP_DEMO = True and rerun this cell when finished.')
else:
    if 'CLOUDFLARED' in globals() and CLOUDFLARED.poll() is None:
        CLOUDFLARED.terminate()
        try:
            CLOUDFLARED.wait(timeout=10)
        except subprocess.TimeoutExpired:
            CLOUDFLARED.kill()
            CLOUDFLARED.wait(timeout=10)
    if 'COMPOSE' in globals() and 'REPO_DIR' in globals():
        subprocess.run(COMPOSE + ['down', '--remove-orphans'], cwd=REPO_DIR, check=True, timeout=120)
        print('Temporary demo stopped. Database volumes retained; no records deleted.')
    else:
        print('No notebook-managed stack to stop. Remote deployment was left running.')
    if 'session' in globals():
        session.close()
        session.headers.pop('Authorization', None)
    globals().pop('ACCESS_TOKEN', None)

## Troubleshooting

- `401`: token missing/expired; obtain a new token and rerun the configuration cell.
- `403`: token lacks the required directory/EMR/patient-flow permissions.
- `400` mentioning hospital ID: check `HOSPITAL_ID` or use the tenant UUID assigned to your deployment.
- Connection/TLS errors: verify the deployment is publicly reachable over HTTPS and that the gateway ingress allows Colab requests.
- `RUN_WRITE_TESTS` requires a role allowed to register patients, create patient journeys, and save clinical notes.